In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
import geojson_validator
from shapely.ops import unary_union
import pandas as pd
import numpy as np
from shapely.geometry import Point, MultiPolygon
import geopandas as gpd
from geopandas import GeoDataFrame
from fuzzywuzzy import process

In [9]:
rc_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\risk-score\Assam\assam_rc_2024-11_reduced.json")
dist_gdf = gpd.read_file(r"D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\assam-flood-expenses\assam_district_2024-11_reduced.json")
expenses_df = pd.read_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\IDS-DRR-Assam\Sources\SDRF\data\flood_expenses_RCgeotagged.csv')

In [3]:
# Function to fix invalid geometries
def fix_geometry(geom):
    if geom is None:
        return None
    if not geom.is_valid:
        # Attempt to fix using buffer(0)
        geom = geom.buffer(0)
    return geom if geom.is_valid else None

# Check and fix geometries
def clean_geometries(gdf):
    # Check for missing or invalid geometries
    gdf['geometry_fixed'] = gdf['geometry'].apply(fix_geometry)

    # Drop rows with irreparable (None) geometries
    gdf = gdf.dropna(subset=['geometry_fixed'])

    return gdf

def geometry_to_text(geom):
    if geom is None:
        return None
    
    # Handle Polygon and LineString geometries
    if geom.geom_type == 'Polygon':
        coords = list(geom.exterior.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    elif geom.geom_type == 'LineString':
        coords = list(geom.coords)
        return str([[lon, lat] for lon, lat in coords])
    
    # Handle MultiPolygon and MultiLineString geometries
    elif geom.geom_type in ['MultiPolygon', 'MultiLineString']:
        all_coords = []
        for part in geom.geoms:  # Loop through each sub-geometry
            if part.geom_type == 'Polygon':
                coords = list(part.exterior.coords)
            else:
                coords = list(part.coords)
            all_coords.append([[lon, lat] for lon, lat in coords])
        return str(all_coords)
    
    # Catch other geometry types if necessary
    else:
        return None

def get_best_match(block_name, block_names):
    match, score = process.extractOne(block_name, block_names)
    return match if score > 80 else None  # Adjust threshold as needed

In [5]:
import json

# Flatten nested geometries if required
fixed_geo_flattened = {
    key: [json.dumps(item) if isinstance(item, dict) else item for item in value]
    for key, value in dist_gdf.items()
}
geo_fixed = pd.DataFrame.from_dict(fixed_geo_flattened)

In [6]:
# Simplify MultiPolygon by selecting the largest Polygon
def simplify_multipolygon(geometry):
    if isinstance(geometry, MultiPolygon):
        # Select the largest Polygon by area
        return max(geometry.geoms, key=lambda geom: geom.area)
    return geometry

# Apply the simplification
dist_gdf['geometry'] = dist_gdf['geometry'].apply(simplify_multipolygon)

geo_fixed = gpd.GeoDataFrame(dist_gdf, geometry='geometry')


In [7]:
dist_gdf

,revenue_ci,revenue_cr,HQ,are_new,dtname,object_id,dtcode11,district_LST_percent_deviation,geometry
0,Sadiya,Sadiya,None,851,TINSUKIA,18-309,18-309,-1.836,"POLYGON ((95.48417 27.24135, 95.50370 27.24863..."
1,Bagribari (Pt),Bagribari (Pt),None,213,DHUBRI,18-301,18-301,4.124,"POLYGON ((89.84312 25.88288, 89.85048 25.87964..."
2,Kalgachia,Kalgachia,None,232,BARPETA,18-303,18-303,1.062,"POLYGON ((91.22210 26.17940, 91.23733 26.18028..."
3,Ujani Majuli,Ujani Majuli,None,322,MAJULI,18-760,18-760,-4.036,"POLYGON ((94.57697 27.16797, 94.57225 27.17411..."
4,Algapur,Algapur,None,156,HAILAKANDI,18-318,18-318,-4.121,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
5,Rangia (Pt),Rangia (Pt),None,23,TAMULPUR,18-816,18-816,0.952,"POLYGON ((91.77912 26.47961, 91.78786 26.47533..."
6,Dimow,Demow,None,478,SIVASAGAR,18-311,18-311,-3.293,"POLYGON ((94.73376 26.93260, 94.73374 26.93261..."
7,Baganpara (Pt),Baganpara (Pt),None,212,BAKSA,18-324,18-324,2.445,"POLYGON ((91.24956 26.52328, 91.26269 26.52165..."
8,Rangia(Pt),Rangia\n(Pt),None,186,KAMRUP,18-321,18-321,0.634,"POLYGON ((91.03041 25.88595, 91.02766 25.86919..."
9,Dhakuakhana (Pt-II),Dhakuakhana (Pt-II),None,9,DHEMAJI,18-308,18-308,-2.893,"POLYGON ((94.26818 27.52094, 94.27895 27.49520..."


In [10]:
merged_gdf = expenses_df.merge(geo_fixed[['geometry','dtname']], right_on=['dtname'], left_on=['DISTRICT_FINALISED'], how='left')
merged_gdf = merged_gdf.drop(columns=['dtname','tender_revenueci_location'])
merged_gdf

,Unnamed: 0,Name of the Scheme,Whether scheme,Whether the scheme is funded by NABARD,Proposal Number,Proposal Date,Issued Number,Valid Upto,Admin Department File Number,Brief nature of the scheme,...,tender_district_title_description,tender_district_location,DISTRICT_FINALISED,tender_villages,tender_block,tender_subdistrict,tender_revenueci,HQ_flag,REVENUE_CIRCLE_FINALISED,geometry
0,5,Administrative Approval for the implementation...,New project,No,AA-05-2019-20-0207,01-07-2019,AA/05_2019-20(I)_16,02-08-2022,RGR(RRR)984/2018,Immediate Measures,...,HAILAKANDI,HAILAKANDI,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
1,159,Temporary Restoration Restoration of Old Hospi...,New project,No,AA-05-2019-20-0643,18-02-2020,AA/05_2019-20(I)_147,28-02-2023,RGR(RRR)1147/2019/61,NaN,...,HAILAKANDI,HAILAKANDI,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
2,160,Temporary Restoration Restoration of Approache...,New project,No,AA-05-2019-20-0644,12-02-2020,AA/05_2019-20(I)_148,28-02-2023,RGR(RRR)1147/2019/74,NaN,...,HAILAKANDI,HAILAKANDI,HAILAKANDI,NaN,LALA,Lala,Lala,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
3,161,Temporary Restoration Restoration of Lalacherr...,New project,No,AA-05-2019-20-0645,12-02-2020,AA/05_2019-20(I)_149,28-02-2023,RGR(RRR)1147/2019/86,NaN,...,HAILAKANDI,HAILAKANDI,HAILAKANDI,KATLICHERRA,KATLICHERRA,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
4,162,Temporary Restoration Restoration of Road from...,New project,No,AA-05-2019-20-0646,12-02-2020,AA/05_2019-20(I)_150,28-02-2023,RGR(RRR)1147/2019/102,NaN,...,HAILAKANDI,HAILAKANDI,HAILAKANDI,ALGAPUR,ALGAPUR,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5628,5572,Restoration of flood damaged approach road ero...,New project,No,AA-05-2024-25-7279,30-07-2024,AA/05_2024-25(I)_5557,31-07-2027,241608,FDR,...,CHIRANG,BONGAIGAON,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
5629,5573,Restoration of flood damaged approach road of ...,New project,No,AA-05-2024-25-7280,30-07-2024,AA/05_2024-25(I)_5556,31-07-2027,241608,FDR,...,CHIRANG,BONGAIGAON,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
5630,5603,Road from No. 2 Thekeraguri to Brahmaputra E&D...,New project,No,AA-05-2024-25-7332,08-10-2024,AA/05_2024-25(I)_5614,02-11-2027,496057,mitigation scheme,...,CACHAR,KARIMGANJ,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None
5631,5604,Road from Kherkata GP Office to Medhisuti cons...,New project,No,AA-05-2024-25-7336,08-10-2024,AA/05_2024-25(I)_5618,02-11-2027,496057,mitigation scheme,...,CACHAR,KARIMGANJ,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None


In [11]:
merged_gdf['polygons'] = merged_gdf['geometry'].apply(geometry_to_text)
merged_gdf

,Unnamed: 0,Name of the Scheme,Whether scheme,Whether the scheme is funded by NABARD,Proposal Number,Proposal Date,Issued Number,Valid Upto,Admin Department File Number,Brief nature of the scheme,...,tender_district_location,DISTRICT_FINALISED,tender_villages,tender_block,tender_subdistrict,tender_revenueci,HQ_flag,REVENUE_CIRCLE_FINALISED,geometry,polygons
0,5,Administrative Approval for the implementation...,New project,No,AA-05-2019-20-0207,01-07-2019,AA/05_2019-20(I)_16,02-08-2022,RGR(RRR)984/2018,Immediate Measures,...,HAILAKANDI,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234...","[[92.42243290805344, 24.254958510347983], [92...."
1,159,Temporary Restoration Restoration of Old Hospi...,New project,No,AA-05-2019-20-0643,18-02-2020,AA/05_2019-20(I)_147,28-02-2023,RGR(RRR)1147/2019/61,NaN,...,HAILAKANDI,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234...","[[92.42243290805344, 24.254958510347983], [92...."
2,160,Temporary Restoration Restoration of Approache...,New project,No,AA-05-2019-20-0644,12-02-2020,AA/05_2019-20(I)_148,28-02-2023,RGR(RRR)1147/2019/74,NaN,...,HAILAKANDI,HAILAKANDI,NaN,LALA,Lala,Lala,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234...","[[92.42243290805344, 24.254958510347983], [92...."
3,161,Temporary Restoration Restoration of Lalacherr...,New project,No,AA-05-2019-20-0645,12-02-2020,AA/05_2019-20(I)_149,28-02-2023,RGR(RRR)1147/2019/86,NaN,...,HAILAKANDI,HAILAKANDI,KATLICHERRA,KATLICHERRA,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234...","[[92.42243290805344, 24.254958510347983], [92...."
4,162,Temporary Restoration Restoration of Road from...,New project,No,AA-05-2019-20-0646,12-02-2020,AA/05_2019-20(I)_150,28-02-2023,RGR(RRR)1147/2019/102,NaN,...,HAILAKANDI,HAILAKANDI,ALGAPUR,ALGAPUR,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.42243 24.25496, 92.41848 24.21234...","[[92.42243290805344, 24.254958510347983], [92...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5628,5572,Restoration of flood damaged approach road ero...,New project,No,AA-05-2024-25-7279,30-07-2024,AA/05_2024-25(I)_5557,31-07-2027,241608,FDR,...,BONGAIGAON,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
5629,5573,Restoration of flood damaged approach road of ...,New project,No,AA-05-2024-25-7280,30-07-2024,AA/05_2024-25(I)_5556,31-07-2027,241608,FDR,...,BONGAIGAON,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
5630,5603,Road from No. 2 Thekeraguri to Brahmaputra E&D...,New project,No,AA-05-2024-25-7332,08-10-2024,AA/05_2024-25(I)_5614,02-11-2027,496057,mitigation scheme,...,KARIMGANJ,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None
5631,5604,Road from Kherkata GP Office to Medhisuti cons...,New project,No,AA-05-2024-25-7336,08-10-2024,AA/05_2024-25(I)_5618,02-11-2027,496057,mitigation scheme,...,KARIMGANJ,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None


In [9]:
merged_gdf = merged_gdf.dropna(subset =['object_id'])

KeyError: ['object_id']

In [27]:
from datetime import date, timedelta, datetime


#snapshot_ld_dist['datetime'] = pd.to_datetime(snapshot_ld_dist['timeperiod'], format='%Y_%m')
merged_gdf['timeperiod'] = merged_gdf['month'].str.replace('_', '-') #+ '-01'

# Step 2: Convert the modified column to datetime format
#snapshot_ld_dist['timeperiod_iso'] = pd.to_datetime(snapshot_ld_dist['timeperiod_iso'], format='%Y-%m-%d')
merged_gdf['timeperiod'] = pd.to_datetime(merged_gdf['timeperiod'], format='%Y-%m')


merged_gdf['timeperiod'] = merged_gdf['timeperiod'].dt.strftime('%Y-%m')

In [28]:
merged_gdf.columns = merged_gdf.columns.str.lower().str.replace(' ', '_')
merged_gdf = merged_gdf.rename(columns={'contract_date_:':'contract_date','bid_validity(days)':'bid_validity_days','tender_value_in_₹':'tender_value_in_rupees'})
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].str.replace(',', '')
merged_gdf['tender_value_in_rupees'] = merged_gdf['tender_value_in_rupees'].astype(float)
merged_gdf = merged_gdf.dropna(subset=['awarded_value', 'tender_value_in_rupees','district_finalised'])
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].str.replace(',', '')
merged_gdf['awarded_value'] = merged_gdf['awarded_value'].astype(float)
merged_gdf

,unnamed:_0,tender_id,tender_externalreference,tender_title,work_description,tender_category,tender_type,form_of_contract,product_category,is_multi_currency_allowed_for_boq,...,district_finalised,tender_villages,tender_block,tender_subdistrict,tender_revenueci,hq_flag,revenue_circle_finalised,geometry,polygons,timeperiod
0,159,2017_DoWR_2083_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal 1,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
1,160,2017_DoWR_2238_1,Hailakandi/SDRF/2016-17/2,Dhaleswari Katakhal Drainage Basin Ph-II,Immediate measures to improvement of Dhaleswar...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,NaN,HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
2,210,2018_DoWR_5796_1,HAILAKANDI/SDRF/2017-18/1,IM at Kalinagar Pk-1,Immediate measures to dyke along l/b of river ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MOHANPUR', 'KALINAGAR'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2018-07
3,372,2018_DoWR_6310_2,HAILAKANDI/RIDF-XXIII/1,A E Measures to Protect Sahabad-Rongpur area,Anti Erosion Measures to Protect Sahabad-Rongp...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'SAHABAD', 'RONGPUR', 'Rongpur'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
4,552,2019_DoWR_11496_1,HAILAKANDI/2018-19/SDRF/II,IM at Matijuri,Immediate measures to Restoration for damages ...,Works,Open Tender,Works,Civil Works,No,...,HAILAKANDI,"'MATIJURI', 'Matijuri'",HAILAKANDI,Hailakandi,Hailakandi,True,Hailakandi,"POLYGON ((92.53425 24.75484, 92.50467 24.70250...","[[92.53424687083123, 24.75483721712959], [92.5...",2019-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2778,2673,2023_SCD_34601_8,Tender/NIT/Pt/2023-24/7736,EARTHEN EMBANKMENT/GUIDE BUND AT MORAN CHUTIA ...,EARTHEN EMBANKMENT/GUIDE BUND AT MORAN CHUTIA ...,Works,Open Tender,Item Rate,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2779,2680,2024_DoWR_36023_2,GOLAGHAT/2023-24/NIDA/I,Anti erosion measures to protect Amguri Basapa...,Anti erosion measures to protect Amguri Basapa...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2780,2682,2024_DoWR_36478_1,DHEMAJI/2023-24/NIDA/II,Extension of Gainadi L/B embankment from Sumon...,Extension of Gainadi L/B embankment from Sumon...,Works,Open Tender,Works,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-08
2781,2775,2024_ICD_38053_1,505342/173 dated 09.07.2024,"Construction of Boundary Wall, Land Developmen...","Construction of Boundary Wall, Land Developmen...",Works,Open Tender,Item Rate,Civil Works,No,...,CONFLICT,NaN,NaN,NaN,NaN,False,NaN,None,None,2024-12


In [12]:
merged_gdf.to_csv(r'D:\CivicDataLab_IDS-DRR\IDS-DRR_Github\Dashboards\assam-flood-expenses\expenses_2019-2024.csv', index=False)